# TrendPulse - Part 1: Python Foundations

## Task 1: Setup

In this cell I set up the file paths, the list of countries and the thresholds.

**Notebook discipline:**
- Every section has a heading.
- Before submitting I will use Runtime -> Restart and run all.
- The notebook must run top to bottom without any error.

## Task 1: Setup

I declared the configuration (countries, file names, thresholds) as variables at the top and print a run header with f-strings. Notebook discipline: clear headings, no hidden state, and the notebook runs top to bottom with Restart and run all.

In [35]:
from datetime import datetime

# File paths
US_FILE = "USvideos.csv"
GB_FILE = "GBvideos.csv"
IN_FILE = "INvideos.csv"
CATEGORIES_FILE = "categories.csv"

# Countries
COUNTRIES = ["US", "GB", "IN"]

# Thresholds (used later in Task 7)
VIRAL_VIEWS = 1000000
GOOD_LIKE_RATIO = 0.90
LOW_VIEWS = 10000

# Run header
print(f"TrendPulse run | started: {datetime.now():%Y-%m-%d %H:%M:%S} | countries: {COUNTRIES} | thresholds: viral={VIRAL_VIEWS}, like_ratio={GOOD_LIKE_RATIO}, low={LOW_VIEWS}")

TrendPulse run | started: 2026-11-05 11:08:28 | countries: ['US', 'GB', 'IN'] | thresholds: viral=1000000, like_ratio=0.9, low=10000


In [36]:
import pandas as pd

print(pd.read_csv(US_FILE).head())
print(pd.read_csv(CATEGORIES_FILE).head())

      video_id trending_date                                  title  \
0  g_N-w8LeAi8      17.14.11           Honest Football Prank LEAKED   
1  Ugyql2NlPAd      17.14.11              Surprising Cricket Review   
2  8fE19jkxJ9n      17.14.11  Mira Solis - Football (Live) OFFICIAL   
3  WOY0nn7HCsO      17.14.11     Zed Rivera - Space (Lyric Video)?!   
4  2Lc_Cs5VVr_      17.14.11             Surprising Election Recipe   

           channel_title  category_id              publish_time  \
0          QuietSpeech10           22  2017-11-14T07:04:28.000Z   
1  IncredibleChallenge41           25  2017-11-12T22:35:48.000Z   
2         WildTutorial42           10  2017-11-13T08:40:22.000Z   
3    ShockingInterview30           10  2017-10-27T22:21:47.000Z   
4         UltimateVlog96           27  2017-11-09T13:20:20.000Z   

              tags   views    likes  dislikes  comment_count  \
0   Monsoon|Robots   48555   2189.0       180          136.0   
1           Winter  198385   1159.0       

In [37]:
def load_country(path, country, max_rows=None):
    df = pd.read_csv(path, nrows=max_rows)
    df["country"] = country
    return df.to_dict("records")

us_records = load_country(US_FILE, "US")
gb_records = load_country(GB_FILE, "GB")
in_records = load_country(IN_FILE, "IN")

print(f"US records: {len(us_records)}")
print(f"GB records: {len(gb_records)}")
print(f"IN records: {len(in_records)}")

US records: 16080
GB records: 15075
IN records: 15075


In [38]:
categories_df = pd.read_csv(CATEGORIES_FILE)
category_names = dict(zip(categories_df["category_id"], categories_df["category_name"]))

print(category_names)

{1: 'Film & Animation', 2: 'Autos & Vehicles', 10: 'Music', 15: 'Pets & Animals', 17: 'Sports', 19: 'Travel & Events', 20: 'Gaming', 22: 'People & Blogs', 23: 'Comedy', 24: 'Entertainment', 25: 'News & Politics', 26: 'Howto & Style', 27: 'Education', 28: 'Science & Technology', 29: 'Nonprofits & Activism', 30: 'Movies', 43: 'Shows'}


In [39]:
import math

all_records = us_records + gb_records + in_records
print("total records:", len(all_records))

fields = ["views", "likes", "dislikes", "comment_count", "trending_date", "publish_time", "description"]

for f in fields:
    blank = 0
    for rec in all_records:
        v = rec[f]
        if v is None or v == "" or (isinstance(v, float) and math.isnan(v)):
            blank += 1
    print(f"{f}: {blank} blank")

total records: 46230
views: 0 blank
likes: 12 blank
dislikes: 0 blank
comment_count: 36 blank
trending_date: 0 blank
publish_time: 0 blank
description: 1624 blank


## Task 2: Loading data
I wrote load_country() to read each CSV file with pandas and return a list of dictionaries, adding a country field. I also built category_names, a dictionary that maps category_id to category_name.

In [40]:
for rec in all_records:
    if rec["description"] is None or rec["description"] != rec["description"]:
        rec["description"] = ""
    for f in ["likes", "comment_count"]:
        v = rec[f]
        if v is None or v == "" or (isinstance(v, float) and math.isnan(v)):
            rec[f] = 0

print("done cleaning")

done cleaning


## Task 3: Cleaning data

I filled blank descriptions with an empty string and replaced missing likes and comment_count values with 0.

I did not drop these rows because only 12 likes and 36 comment_count values were blank out of 46,230 records, and dropping them would lose valid views and titles. A blank count is treated as 0.

In [41]:
from datetime import datetime

fail_counts = {"views": 0, "likes": 0, "dislikes": 0, "comment_count": 0,
               "trending_date": 0, "publish_time": 0}

def to_int(v):
    return int(float(v))

def to_bool(v):
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() == "true"

for rec in all_records:
    # numbers -> int
    for f in ["views", "likes", "dislikes", "comment_count"]:
        try:
            rec[f] = to_int(rec[f])
        except (ValueError, TypeError):
            rec[f] = 0
            fail_counts[f] += 1

    # flags -> bool
    for f in ["comments_disabled", "ratings_disabled", "video_error_or_removed"]:
        rec[f] = to_bool(rec[f])

    # trending_date: yy.dd.mm
    try:
        rec["trending_date"] = datetime.strptime(rec["trending_date"], "%y.%d.%m").date()
    except (ValueError, TypeError):
        rec["trending_date"] = None
        fail_counts["trending_date"] += 1

    # publish_time: ISO format
    try:
        rec["publish_time"] = datetime.strptime(rec["publish_time"], "%Y-%m-%dT%H:%M:%S.%fZ")
    except (ValueError, TypeError):
        rec["publish_time"] = None
        fail_counts["publish_time"] += 1

print("Parse failures per field:")
for f, c in fail_counts.items():
    print(f"{f}: {c} failed")

print(all_records[0]["trending_date"], all_records[0]["publish_time"])
print(type(all_records[0]["views"]), type(all_records[0]["likes"]))

Parse failures per field:
views: 0 failed
likes: 0 failed
dislikes: 0 failed
comment_count: 0 failed
trending_date: 0 failed
publish_time: 0 failed
2017-11-14 2017-11-14 07:04:28
<class 'int'> <class 'int'>


## Task 4: Core data structures

In [42]:
# cleaned records are already a list of dictionaries: all_records

# tuples for key fields (immutable)
key_tuples = [(r["video_id"], r["trending_date"], r["views"], r["likes"]) for r in all_records]
print("first tuple:", key_tuples[0])

# sets of unique video IDs per country
countries = ["US", "GB", "IN"]
ids_by_country = {}
for c in countries:
    ids_by_country[c] = set(r["video_id"] for r in all_records if r["country"] == c)
    print(f"{c}: {len(ids_by_country[c])} unique videos")

# intersection / difference
in_all_three = ids_by_country["US"] & ids_by_country["GB"] & ids_by_country["IN"]
in_us_and_gb = ids_by_country["US"] & ids_by_country["GB"]
only_us = ids_by_country["US"] - ids_by_country["GB"] - ids_by_country["IN"]
print("in all 3 countries:", len(in_all_three))
print("in both US and GB:", len(in_us_and_gb))
print("only in US:", len(only_us))

first tuple: ('g_N-w8LeAi8', datetime.date(2017, 11, 14), 48555, 2189)
US: 5068 unique videos
GB: 4113 unique videos
IN: 5878 unique videos
in all 3 countries: 20
in both US and GB: 134
only in US: 4834


In [43]:
channel_counts = {}
for r in all_records:
    ch = r["channel_title"]
    if ch in channel_counts:
        channel_counts[ch] += 1
    else:
        channel_counts[ch] = 1

top_channels = sorted(channel_counts.items(), key=lambda x: x[1], reverse=True)[:15]
print("Top 15 channels by trending appearances:")
for name, count in top_channels:
    print(f"{name}: {count}")

Top 15 channels by trending appearances:
QuietChallenge75: 2464
OfficialReview41: 1586
UltimateUnboxing65: 1433
InsaneReaction46: 1233
Brand NewChallenge45: 1175
CrazyTrailer91: 1018
FinalTrailer17: 911
UltimateSpeech20: 770
Brand NewSecret51: 684
WildHighlights7: 682
SurprisingHighlights41: 650
FinalBreakdown11: 643
CrazyRecipe67: 603
QuietInterview9: 570
UltimateVlog96: 565


In [44]:
summary = {}
for r in all_records:
    country = r["country"]
    cat = category_names.get(str(r["category_id"]), category_names.get(r["category_id"], "Unknown"))
    if country not in summary:
        summary[country] = {}
    summary[country][cat] = summary[country].get(cat, 0) + 1

for country, cats in summary.items():
    print(f"{country}:")
    for cat, n in list(cats.items())[:5]:
        print(f"   {cat}: {n} records")

US:
   People & Blogs: 1423 records
   News & Politics: 755 records
   Music: 3052 records
   Education: 561 records
   Entertainment: 4371 records
GB:
   Science & Technology: 470 records
   Howto & Style: 588 records
   Comedy: 941 records
   Entertainment: 2844 records
   Music: 5013 records
IN:
   Music: 2298 records
   Film & Animation: 770 records
   Entertainment: 5117 records
   Comedy: 966 records
   People & Blogs: 1098 records


**Why each structure:**
- **List of dictionaries** keeps every record with named fields, and a list can be looped over and changed, so it is mutable.
- **Tuples** hold the key fields (video_id, trending_date, views, likes) because a record's identity should not change after cleaning. Tuples are immutable.
- **Sets** give unique video IDs per country and make intersection and difference fast, for videos trending in more than one country or in only one. Sets are mutable.
- **Dictionary counter** counts appearances per channel with quick lookup by channel name. Dictionaries are mutable.
- **Slicing** `[:15]` on the sorted list shows the top 15 channels.
- **Nested dictionary** (country -> category name -> count) matches the natural grouping of the data. It is mutable.

## Task 5: Scope

`outside_var` is a global variable, so a function can read it. `inside_var` is a local variable that exists only while the function runs. Trying to use `inside_var` outside the function raises a `NameError`, which shows that a local variable cannot be seen from the outside.

In [45]:
outside_var = "I am defined outside the function"

def scope_demo():
    inside_var = "I am defined inside the function"
    print("Inside function, can read outside_var:", outside_var)
    print("Inside function, inside_var:", inside_var)

scope_demo()
print("Outside function, outside_var:", outside_var)

try:
    print(inside_var)
except NameError as e:
    print("Error when accessing inside_var outside the function:", e)

Inside function, can read outside_var: I am defined outside the function
Inside function, inside_var: I am defined inside the function
Outside function, outside_var: I am defined outside the function
Error when accessing inside_var outside the function: name 'inside_var' is not defined


## Task 6: Loops over strings and sequences

`title_features(title)` loops over the characters and words of a title to count uppercase words, exclamation marks, question marks and the title length in words. A `for` loop with `range()` and index access is used for the words. A `while` loop scans the records until it finds the first video above a view threshold, and `break` and `continue` are used in the loops below.

In [46]:
def title_features(title):
    upper_words = 0
    exclaim = 0
    question = 0

    # for loop over characters
    for ch in title:
        if ch == "!":
            exclaim += 1
        elif ch == "?":
            question += 1

    # for loop with range() and index access over words
    words = title.split()
    for i in range(len(words)):
        w = words[i]
        if w.isupper() and len(w) > 1:
            upper_words += 1

    return {
        "uppercase_words": upper_words,
        "exclamation_marks": exclaim,
        "question_marks": question,
        "length_in_words": len(words),
    }

print(title_features(all_records[0]["title"]))
print(title_features("WOW!! Is THIS real?"))

{'uppercase_words': 1, 'exclamation_marks': 0, 'question_marks': 0, 'length_in_words': 4}
{'uppercase_words': 2, 'exclamation_marks': 2, 'question_marks': 1, 'length_in_words': 4}


In [47]:
VIEW_THRESHOLD = 1000000

# while loop: scan until the first video above the threshold
i = 0
found = None
while i < len(all_records):
    if all_records[i]["views"] > VIEW_THRESHOLD:
        found = all_records[i]
        break
    i += 1

if found:
    print(f"First video above {VIEW_THRESHOLD} views: {found['title']} ({found['views']} views) at index {i}")
else:
    print("No video found above the threshold")

# for loop with continue: skip records with comments disabled, count the rest with a missing title
skipped = 0
no_title = 0
for r in all_records:
    if r["comments_disabled"]:
        skipped += 1
        continue
    if not r["title"]:
        no_title += 1

print("Skipped (comments disabled):", skipped)
print("Records with comments enabled but no title:", no_title)

First video above 1000000 views: Official Cats Speech EXCLUSIVE (2841278 views) at index 7
Skipped (comments disabled): 1048
Records with comments enabled but no title: 0


## Task 7: Rule-based classification

`classify_video(record)` assigns one label to each record using if/elif/else, boolean logic and nested conditions.

**Thresholds:**
- Viral Hit: views >= 1,000,000 and like ratio >= 0.90
- Steady Performer: views >= 100,000 and like ratio >= 0.80
- Divisive: like ratio < 0.60 (with at least 10 likes + dislikes)
- Low Engagement: everything else that has ratings
- Unrated: ratings are disabled, or likes + dislikes is 0

**Edge cases:** zero views, zero likes and dislikes, ratings disabled and missing values all go to Unrated or Low Engagement first, so there is no division by zero. The like ratio is only calculated when likes + dislikes is greater than 0.

**Hand-trace:** a record with views = 2,500,000, likes = 90,000, dislikes = 5,000 and ratings enabled. ratings_disabled is False, so the first check is skipped. likes + dislikes = 95,000, which is above 0, so like ratio = 90,000 / 95,000 = 0.947. Views are at least 1,000,000 and the ratio is at least 0.90, so the label is **Viral Hit**.

In [48]:
def classify_video(record):
    views = record["views"]
    likes = record["likes"]
    dislikes = record["dislikes"]
    total = likes + dislikes

    # edge cases first, so we never divide by zero
    if record["ratings_disabled"] or total == 0:
        return "Unrated"

    ratio = likes / total

    if views >= 1000000 and ratio >= 0.90:
        return "Viral Hit"
    elif views >= 100000 and ratio >= 0.80:
        return "Steady Performer"
    elif ratio < 0.60 and total >= 10:
        return "Divisive"
    else:
        return "Low Engagement"

label_counts = {}
for r in all_records:
    label = classify_video(r)
    label_counts[label] = label_counts.get(label, 0) + 1

print("Records per label:")
for label, n in label_counts.items():
    print(f"{label}: {n}")

# hand-trace example check
example = {"views": 2500000, "likes": 90000, "dislikes": 5000, "ratings_disabled": False}
print("Hand-trace example ->", classify_video(example))

Records per label:
Low Engagement: 9743
Steady Performer: 26816
Divisive: 263
Viral Hit: 8864
Unrated: 544
Hand-trace example -> Viral Hit


## Task 8: Refactor

I wrote the same counting pattern (`d[key] = d.get(key, 0) + 1`) in several places: channel counts, category counts and label counts. I moved it into one reusable function, `count_by(records, key_func)`. This removed the repeated loops, so the code is shorter and easier to read, and a fix to the counting logic now needs to be made in only one place.

In [49]:
def count_by(records, key_func):
    counts = {}
    for r in records:
        key = key_func(r)
        counts[key] = counts.get(key, 0) + 1
    return counts

# same results as before, using the one function
channel_counts_2 = count_by(all_records, lambda r: r["channel_title"])
label_counts_2 = count_by(all_records, classify_video)
country_counts = count_by(all_records, lambda r: r["country"])

print("Channels counted:", len(channel_counts_2))
print("Same as earlier channel_counts:", channel_counts_2 == channel_counts)
print("Same as earlier label_counts:", label_counts_2 == label_counts)
print("Records per country:", country_counts)

Channels counted: 438
Same as earlier channel_counts: True
Same as earlier label_counts: True
Records per country: {'US': 16080, 'GB': 15075, 'IN': 15075}
